**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 Session 6: 주요 LLM 모델 비교 & Ollama 로컬 서빙

## 📋 학습 목표

1️⃣ 주요 오픈소스 LLM 모델들의 특징과 차이점을 이해한다  
2️⃣ Ollama를 설치하고 로컬에서 LLM을 서빙하는 방법을 익힌다  
3️⃣ **모델 크기별(1.5B vs 3B vs 7B) 품질과 속도를 직접 비교**한다  
4️⃣ sLLM의 강점과 약점을 파악하고, 개선 방향(RAG/파인튜닝)을 이해한다

---

### 🖥️ 실습 환경
- **GPU**: NVIDIA RTX 4060 (8GB VRAM)
- **비교 모델**: Qwen2.5:1.5b / Qwen2.5:3b / Qwen2.5:7b (Q4 양자화)
- **핵심 질문**: 모델 크기가 커지면 얼마나 나아지는가?

## 1️⃣ GPU 환경 점검

먼저 GPU 상태를 확인하고, 메모리 모니터링 함수를 정의합니다.

In [ ]:
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print("⚠️ GPU를 사용할 수 없습니다.")

# GPU 정보 출력
if torch.cuda.is_available():
    print(f"✅ GPU 이름: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("초기 상태")
else:
    print("⚠️ CUDA GPU를 찾을 수 없습니다.")

## 2️⃣ 주요 오픈소스 LLM 모델 비교

현재 가장 많이 사용되는 오픈소스 LLM들을 비교해봅니다.

| 모델 | 개발사 | 크기 | 라이선스 | 특징 |
|------|--------|------|----------|------|
| **Qwen2.5** | Alibaba | 0.5B~72B | Apache 2.0 | 다국어, 한국어 우수 |
| **Llama 3** | Meta | 8B~70B | Llama License | 영어 중심, 범용 성능 |
| **Gemma 2** | Google | 2B~27B | Gemma License | 효율적 아키텍처 |
| **Mistral** | Mistral AI | 7B~8x22B | Apache 2.0 | MoE 아키텍처 |
| **Phi-3** | Microsoft | 3.8B~14B | MIT | 소형 모델 강자 |

### 🔑 RTX 4060 (8GB VRAM)에서의 실행 가능 여부

| 모델 크기 | FP16 | 4bit 양자화 | 추론 가능? |
|-----------|------|-------------|------------|
| 1.5B | ~3GB | ~1GB | ✅ 여유 |
| 3B | ~6GB | ~2GB | ✅ 가능 |
| 7B | ~14GB | ~4GB | ⚠️ 양자화 필수 |
| 13B | ~26GB | ~8GB | ❌ 불가 |

In [ ]:
# 📊 모델 크기별 메모리 요구사항 시각화
print("📊 모델 크기별 예상 VRAM 사용량 (FP16 vs 4bit)")
print("=" * 60)

models = [
    ("Qwen2.5-0.5B", 0.5, 0.3),
    ("Qwen2.5-1.5B", 3.0, 1.0),
    ("Qwen2.5-3B",   6.0, 2.0),
    ("Qwen2.5-7B",  14.0, 4.0),
    ("Llama-3-8B",  16.0, 5.0),
    ("Qwen2.5-14B", 28.0, 8.0),
]

vram_limit = 8.0  # RTX 4060

for name, fp16, q4 in models:
    fp16_bar = "█" * int(fp16 * 2)
    q4_bar = "█" * int(q4 * 2)
    fp16_status = "✅" if fp16 <= vram_limit else "❌"
    q4_status = "✅" if q4 <= vram_limit else "❌"
    print(f"\n🔹 {name}")
    print(f"  FP16: {fp16_bar} {fp16:.1f}GB {fp16_status}")
    print(f"  4bit: {q4_bar} {q4:.1f}GB {q4_status}")

print(f"\n{'=' * 60}")
print(f"📌 RTX 4060 VRAM 한계: {vram_limit}GB")

## 3️⃣ Ollama 설치 및 설정

**Ollama**는 로컬에서 LLM을 쉽게 실행할 수 있게 해주는 도구입니다.

### 📦 Ollama의 장점
- 🔹 한 줄 명령어로 모델 다운로드 및 실행
- 🔹 자동 양자화 (GGUF 포맷)
- 🔹 OpenAI 호환 REST API 제공
- 🔹 GPU 자동 감지 및 활용

In [ ]:
import subprocess
import shutil

# 🔍 Ollama 설치 여부 확인
ollama_path = shutil.which("ollama")

if ollama_path:
    print(f"✅ Ollama가 이미 설치되어 있습니다: {ollama_path}")
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"📌 버전: {result.stdout.strip()}")
else:
    print("⚠️ Ollama가 설치되어 있지 않습니다.")
    print("\n📥 설치 방법 (터미널에서 실행):")
    print("  Linux: curl -fsSL https://ollama.ai/install.sh | sh")
    print("  macOS: brew install ollama")
    print("  Windows: https://ollama.ai 에서 다운로드")

In [ ]:
# 🚀 Ollama 서버 상태 확인
import requests
import json

OLLAMA_BASE_URL = "http://localhost:11434"

def check_ollama_server():
    """Ollama 서버가 실행 중인지 확인"""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get("models", [])
            print("✅ Ollama 서버가 실행 중입니다!")
            if models:
                print(f"\n📋 설치된 모델 목록 ({len(models)}개):")
                for m in models:
                    size_gb = m.get('size', 0) / (1024**3)
                    print(f"  🔹 {m['name']} ({size_gb:.1f}GB)")
            else:
                print("📋 설치된 모델이 없습니다.")
            return True
        return False
    except requests.ConnectionError:
        print("❌ Ollama 서버에 연결할 수 없습니다.")
        print("\n🔧 서버 시작 방법 (터미널에서):")
        print("  ollama serve")
        return False

server_running = check_ollama_server()

## 4️⃣ 모델 다운로드 (1.5B / 3B / 7B)

3가지 크기의 모델을 다운로드하여 **직접 비교**합니다.

### 📌 모델별 특성 (RTX 4060 기준)
| 모델 | 크기 | VRAM 사용 | 속도 | 품질 |
|------|------|-----------|------|------|
| `qwen2.5:1.5b` | ~1GB | ~2GB | 매우 빠름 | 기본 |
| `qwen2.5:3b` | ~2GB | ~3.5GB | 빠름 | 양호 |
| `qwen2.5:7b` | ~4.5GB | ~6.5GB | 보통 | 우수 |

> 💡 7B 모델은 Ollama가 자동으로 Q4 양자화된 GGUF를 사용하므로 RTX 4060에서도 추론 가능합니다.

In [ ]:
import subprocess
import time

def pull_model(model_name):
    """Ollama 모델을 다운로드합니다."""
    print(f"📥 모델 다운로드 시작: {model_name}")
    print("⏳ 네트워크 속도에 따라 시간이 걸릴 수 있습니다...")
    
    start_time = time.time()
    
    try:
        result = subprocess.run(
            ["ollama", "pull", model_name],
            capture_output=True, text=True, timeout=600
        )
        
        elapsed = time.time() - start_time
        
        if result.returncode == 0:
            print(f"✅ 다운로드 완료! ({elapsed:.1f}초 소요)")
        else:
            print(f"❌ 다운로드 실패: {result.stderr}")
    except subprocess.TimeoutExpired:
        print("⏰ 타임아웃! 터미널에서 직접 실행해주세요:")
        print(f"  ollama pull {model_name}")
    except FileNotFoundError:
        print("❌ ollama 명령어를 찾을 수 없습니다. 설치를 먼저 진행해주세요.")

# 🔹 3가지 크기의 모델 다운로드
models = ["qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b"]

for model in models:
    pull_model(model)
    print()

In [ ]:
# 📋 다운로드된 모델 확인
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    print("📋 현재 설치된 Ollama 모델:")
    print(result.stdout)
except FileNotFoundError:
    print("❌ ollama 명령어를 찾을 수 없습니다.")

## 5️⃣ Ollama 로컬 추론 실습

Ollama의 REST API를 사용하여 로컬 모델에 질의합니다.

### 📡 Ollama API 엔드포인트
- `POST /api/generate` - 텍스트 생성
- `POST /api/chat` - 대화형 생성
- `GET /api/tags` - 설치된 모델 목록

In [ ]:
import requests
import json
import time

def ollama_generate(model, prompt, temperature=0.7, max_tokens=512):
    """Ollama /api/generate 엔드포인트로 텍스트 생성"""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
        }
    }
    
    start_time = time.time()
    response = requests.post(url, json=payload, timeout=120)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        return {
            "response": result["response"],
            "elapsed": elapsed,
            "eval_count": result.get("eval_count", 0),
            "eval_duration": result.get("eval_duration", 0),
        }
    else:
        return {"error": response.text}

# 🔹 간단한 추론 테스트
print("🚀 Ollama 로컬 추론 테스트")
print("=" * 50)

result = ollama_generate("qwen2.5:1.5b", "한국의 수도는 어디인가요? 간단히 답해주세요.")

if "error" not in result:
    print(f"📝 응답: {result['response']}")
    print(f"⏱️ 소요 시간: {result['elapsed']:.2f}초")
    if result['eval_count'] > 0:
        tokens_per_sec = result['eval_count'] / (result['eval_duration'] / 1e9)
        print(f"🔢 생성 토큰: {result['eval_count']}개")
        print(f"⚡ 처리 속도: {tokens_per_sec:.1f} tokens/sec")
else:
    print(f"❌ 에러: {result['error']}")

In [ ]:
# 💬 대화형 API (Chat) 사용하기
def ollama_chat(model, messages, temperature=0.7, max_tokens=512):
    """Ollama /api/chat 엔드포인트로 대화형 생성"""
    url = f"{OLLAMA_BASE_URL}/api/chat"
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
        }
    }
    
    start_time = time.time()
    response = requests.post(url, json=payload, timeout=120)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        return {
            "response": result["message"]["content"],
            "elapsed": elapsed,
            "eval_count": result.get("eval_count", 0),
            "eval_duration": result.get("eval_duration", 0),
        }
    else:
        return {"error": response.text}

# 🔹 Chat API 테스트
print("💬 Ollama Chat API 테스트")
print("=" * 50)

messages = [
    {"role": "system", "content": "당신은 친절한 한국어 AI 어시스턴트입니다."},
    {"role": "user", "content": "머신러닝과 딥러닝의 차이를 간단히 설명해주세요."}
]

result = ollama_chat("qwen2.5:1.5b", messages)

if "error" not in result:
    print(f"📝 응답:\n{result['response']}")
    print(f"\n⏱️ 소요 시간: {result['elapsed']:.2f}초")
else:
    print(f"❌ 에러: {result['error']}")

## 6️⃣ 모델 크기별 비교 벤치마크 (1.5B vs 3B vs 7B)

같은 질문을 3개 모델에 던져서 **품질과 속도 차이**를 직접 확인합니다.

### 📊 비교 항목
- 🔹 **응답 품질**: 사실 정확성, 한국어 자연스러움, 환각 여부
- 🔹 **응답 속도**: tokens/sec
- 🔹 **핵심 관찰**: 모델 크기가 커질수록 어떤 영역이 개선되는가?

In [ ]:
# 📊 벤치마크용 테스트 질문들
# 다양한 능력을 테스트하는 질문 구성
benchmark_questions = [
    {
        "category": "사실 지식",
        "question": "대한민국의 역대 대통령을 시간순으로 나열해주세요.",
        "note": "환각(hallucination) 여부 확인 — 모델 크기가 커질수록 정확해지는가?"
    },
    {
        "category": "추론",
        "question": "A가 B보다 키가 크고, B가 C보다 키가 큽니다. C가 D보다 키가 크다면, 가장 키가 큰 사람은 누구인가요?",
        "note": "논리적 추론 능력 비교"
    },
    {
        "category": "코드 생성",
        "question": "Python으로 피보나치 수열의 n번째 항을 구하는 함수를 작성해주세요.",
        "note": "sLLM이 비교적 잘하는 영역 — 크기별 차이가 적을 수 있음"
    },
    {
        "category": "한국어 이해",
        "question": "'빛이 좋다'와 '빚이 좋다'의 의미 차이를 설명해주세요.",
        "note": "한국어 뉘앙스 이해 능력 비교"
    },
]

print(f"📋 벤치마크 질문 {len(benchmark_questions)}개 준비 완료")
for i, q in enumerate(benchmark_questions, 1):
    print(f"  {i}. [{q['category']}] {q['question'][:40]}...")
    print(f"     → 관찰 포인트: {q['note']}")

In [ ]:
# 🏃 3개 모델 벤치마크 실행
models_to_test = ["qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b"]

# 설치된 모델 확인
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    if response.status_code == 200:
        installed = [m["name"] for m in response.json().get("models", [])]
        print(f"📋 설치된 모델: {installed}")
        available = [m for m in models_to_test if any(m in inst for inst in installed)]
        missing = [m for m in models_to_test if m not in available]
        if missing:
            print(f"⚠️ 미설치 모델: {missing} — 위의 다운로드 셀을 먼저 실행하세요")
        models_to_test = available
        print(f"🎯 테스트할 모델: {models_to_test}")
except:
    print("⚠️ Ollama 서버에 연결할 수 없습니다. 'ollama serve'를 먼저 실행하세요.")

benchmark_results = []

for model in models_to_test:
    print(f"\n{'=' * 60}")
    print(f"🔄 모델: {model} 벤치마크 시작")
    print("=" * 60)
    
    model_results = []
    
    for q in benchmark_questions:
        print(f"\n  📝 [{q['category']}] 테스트 중...")
        
        messages = [
            {"role": "system", "content": "간결하고 정확하게 한국어로 답변해주세요."},
            {"role": "user", "content": q["question"]}
        ]
        
        result = ollama_chat(model, messages, temperature=0.3, max_tokens=256)
        
        if "error" not in result:
            tokens_per_sec = 0
            if result['eval_count'] > 0 and result['eval_duration'] > 0:
                tokens_per_sec = result['eval_count'] / (result['eval_duration'] / 1e9)
            
            model_results.append({
                "category": q["category"],
                "elapsed": result["elapsed"],
                "tokens": result["eval_count"],
                "tps": tokens_per_sec,
                "response": result["response"],
            })
            
            print(f"    ⏱️ {result['elapsed']:.2f}초 | {tokens_per_sec:.1f} tok/s")
            print(f"    💬 {result['response'][:100]}...")
        else:
            print(f"    ❌ 에러: {result['error'][:50]}")
    
    benchmark_results.append({
        "model": model,
        "results": model_results
    })

print(f"\n✅ 벤치마크 완료!")

In [ ]:
# 📊 모델 크기별 비교 결과 (나란히 비교)
print("📊 모델 크기별 비교 결과")
print("=" * 70)

# 카테고리별로 모델 간 응답을 나란히 비교
categories = [q["category"] for q in benchmark_questions]

for cat in categories:
    print(f"\n{'─' * 70}")
    print(f"📌 [{cat}]")
    print(f"{'─' * 70}")
    
    for br in benchmark_results:
        model = br["model"]
        for r in br["results"]:
            if r["category"] == cat:
                print(f"\n  🔹 {model} ({r['tps']:.1f} tok/s, {r['elapsed']:.1f}초)")
                # 응답을 200자까지 표시
                resp_preview = r["response"][:200].replace('\n', '\n    ')
                print(f"    {resp_preview}")

# 속도 요약
print(f"\n{'=' * 70}")
print("⚡ 속도 요약")
print(f"{'=' * 70}")
for br in benchmark_results:
    model = br["model"]
    results = br["results"]
    if results:
        avg_tps = sum(r["tps"] for r in results) / len(results)
        avg_elapsed = sum(r["elapsed"] for r in results) / len(results)
        print(f"  {model:20s} | 평균 {avg_tps:.1f} tok/s | 평균 {avg_elapsed:.1f}초")

## 7️⃣ Ollama OpenAI 호환 API

Ollama는 **OpenAI 호환 API**를 제공하여, 기존 OpenAI 코드를 거의 그대로 사용할 수 있습니다.

In [ ]:
# 🔄 OpenAI 호환 API 사용법
# pip install openai
from openai import OpenAI

# Ollama의 OpenAI 호환 엔드포인트
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama는 API 키 불필요 (아무 값이나 가능)
)

print("🔄 OpenAI 호환 API로 Ollama 호출")
print("=" * 50)

try:
    response = client.chat.completions.create(
        model="qwen2.5:1.5b",
        messages=[
            {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다."},
            {"role": "user", "content": "Python의 리스트 컴프리헨션을 예제와 함께 설명해주세요."}
        ],
        temperature=0.7,
        max_tokens=300,
    )
    
    print(f"📝 응답:\n{response.choices[0].message.content}")
    print(f"\n📊 사용 토큰: {response.usage}")
except Exception as e:
    print(f"❌ 에러: {e}")
    print("\n💡 Ollama 서버가 실행 중인지 확인해주세요.")

## 8️⃣ 스트리밍 출력

긴 응답의 경우, 스트리밍으로 실시간 출력을 받을 수 있습니다.

In [ ]:
# 🌊 스트리밍 출력 예제
import requests
import json

def ollama_stream(model, prompt):
    """스트리밍 방식으로 Ollama 응답을 받습니다."""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {"num_predict": 200}
    }
    
    print(f"🌊 스트리밍 시작 (모델: {model})")
    print("-" * 40)
    
    full_response = ""
    start_time = time.time()
    
    with requests.post(url, json=payload, stream=True, timeout=120) as resp:
        for line in resp.iter_lines():
            if line:
                data = json.loads(line)
                token = data.get("response", "")
                print(token, end="", flush=True)
                full_response += token
                
                if data.get("done", False):
                    break
    
    elapsed = time.time() - start_time
    print(f"\n\n{'=' * 40}")
    print(f"⏱️ 총 소요 시간: {elapsed:.2f}초")
    print(f"📝 총 문자 수: {len(full_response)}")

# 스트리밍 테스트
try:
    ollama_stream("qwen2.5:1.5b", "파이썬으로 간단한 웹 크롤러를 만드는 방법을 알려주세요.")
except Exception as e:
    print(f"❌ 에러: {e}")

## 9️⃣ 모델 관리 명령어

Ollama에서 자주 사용하는 모델 관리 명령어들을 정리합니다.

In [ ]:
# 📖 Ollama 주요 명령어 정리
commands = [
    ("ollama list", "설치된 모델 목록 확인"),
    ("ollama pull qwen2.5:1.5b", "모델 다운로드"),
    ("ollama run qwen2.5:1.5b", "모델 실행 (대화형)"),
    ("ollama show qwen2.5:1.5b", "모델 정보 확인"),
    ("ollama rm qwen2.5:1.5b", "모델 삭제"),
    ("ollama cp qwen2.5:1.5b my-model", "모델 복사"),
    ("ollama serve", "Ollama 서버 시작"),
    ("OLLAMA_HOST=0.0.0.0 ollama serve", "외부 접근 허용"),
]

print("📖 Ollama 주요 명령어")
print("=" * 60)
for cmd, desc in commands:
    print(f"  🔹 {cmd}")
    print(f"     └─ {desc}")
    print()

In [ ]:
# 🧹 GPU 메모리 확인 (마무리)
print_gpu_memory("실습 종료")

print("\n📌 오늘의 핵심 정리:")
print("  1️⃣ Ollama로 로컬에서 LLM을 쉽게 서빙할 수 있다")
print("  2️⃣ REST API와 OpenAI 호환 API 두 가지 방식으로 호출 가능")
print("  3️⃣ 모델 크기가 커질수록 품질(정확성, 한국어)이 개선된다")
print("  4️⃣ 코드 생성, 간단한 추론 등은 sLLM(1.5B~3B)도 충분히 활용 가능")
print("  5️⃣ 사실 지식이나 한국어 품질이 부족한 영역은 → RAG(Part 2)나 파인튜닝(Part 4)으로 개선")

print("\n🔑 sLLM 활용 전략:")
print("  ✅ 잘하는 것: 코드 생성, 형식 변환, 간단한 추론, 요약")
print("  ⚠️ 부족한 것: 사실 기반 지식, 복잡한 한국어, 전문 도메인")
print("  💡 개선 방법: RAG로 지식 보강 / 파인튜닝으로 도메인 특화")

---

## 🎯 실습 과제

1️⃣ 벤치마크 결과를 보고, 각 카테고리에서 **어떤 크기의 모델이 가장 정확했는지** 정리해보세요  
2️⃣ 자신만의 벤치마크 질문 3개를 추가하여 테스트해보세요 (예: 번역, 감정 분석, 수학 문제)  
3️⃣ **토론**: 여러분의 업무/프로젝트에서 sLLM을 활용한다면, 어떤 크기의 모델이 적합할까요? 속도 vs 품질 트레이드오프를 고려해보세요.

---

## 📚 참고 자료
- [Ollama 공식 문서](https://ollama.ai)
- [Ollama GitHub](https://github.com/ollama/ollama)
- [Qwen2.5 모델 카드](https://huggingface.co/Qwen)